# Lab 10: check Gemini's labels, then use them

**50–60 minutes.** Work with your team. Make a copy named `lab10_mine.ipynb`. Switch who types halfway through.

Most comments in r/AmItheAsshole start with a verdict: NTA (not the asshole) or YTA (you're the asshole). Some comments have no verdict word. The question in this lab: does a comment say the poster was in the wrong?

For comments with a verdict, the verdict gives the answer, so you can check Gemini's labels against it. Then you use Gemini on comments with no verdict, where nothing else gives the answer.

You need your Gemini key in the `.env` file. The lab makes 40 calls, and 20 more if you do the last part.

In [ ]:
from pathlib import Path
import json
import logging
import os
import time

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

for folder in (Path.cwd(), *Path.cwd().parents):
    env_path = folder / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break

key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"

logging.getLogger("google_genai.types").setLevel(logging.ERROR)  # a warning google-genai prints on every call
pd.set_option("display.max_colwidth", 300)  # show more of each text
print("Key loaded.")

## 1. Load the comments and set a baseline

In [ ]:
url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
comments = pd.read_csv(url).rename(columns={"idstr": "id"})
comments["verdict"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")[0]
comments["verdict"].value_counts(dropna=False)  # Check: NTA 657, YTA 378, NaN 477

Gemini should not see the verdict words, or the check would be too easy. Remove NTA, YTA, ESH, NAH and INFO wherever they appear in a comment, and save the result in `comments["text"]`.

*Hint: in a regular expression, `|` means "or".*

In [ ]:
comments["text"] = comments["body"].str.replace(r"\b(____)\b", "", regex=True, case=False)
comments["text"] = comments["text"].str.split().str.join(" ").str.lstrip(" .,:;!-–—")  # remove extra spaces and the punctuation left at the start
comments["text"].str.contains(r"\b(?:NTA|YTA|ESH|NAH|INFO)\b", case=False).sum()  # comments that still contain a verdict word. Check: 0

In [ ]:
# Comments that start with NTA or YTA. The verdict gives the answer: YTA means yes, the poster was in the wrong
with_verdict = comments.dropna(subset=["verdict"]).copy()
with_verdict["answer"] = with_verdict["verdict"].map({"YTA": "yes", "NTA": "no"})

# Comments with no verdict word anywhere. Nothing gives the answer
no_verdict = comments[~comments["body"].str.contains(r"\b(?:NTA|YTA|ESH|NAH|INFO)\b", case=False)]
len(with_verdict), len(no_verdict)  # Check: 1035 and 176

In [ ]:
# The accuracy of always guessing "no" on with_verdict. Check: 0.63

## 2. A keyword rule

A keyword rule is free and gives the same answer every time, so you can check it on all 1,035 comments with a verdict. This one says yes if a comment contains "rude", "selfish", "should have", or a word that starts with "apologi".

In [ ]:
words = r"\b(?:rude|selfish|should have|apologi\w*)\b"
with_verdict["rule"] = with_verdict["text"].str.contains(words, case=False).map({True: "yes", False: "no"})
with_verdict["rule"].value_counts()  # Check: no 948, yes 87

In [ ]:
# The accuracy of the rule. Check: 0.61

In [ ]:
pd.crosstab(with_verdict["answer"], with_verdict["rule"])  # the answer in rows, the rule's label in columns

The rule is less accurate than always guessing "no". Which mistake does it make more often: false positives (answer no, rule yes) or false negatives (answer yes, rule no)?

Your answer:

## 3. Check Gemini on 20 comments with a verdict

Gemini labels the 20 comments from Monday's file, 10 NTA and 10 YTA. The instruction does not mention verdicts, so you can use the same one on comments with no verdict in part 5. Read the cells, then run them.

In [ ]:
SYSTEM = """You read comments on posts in the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong.

Label whether the comment says the poster was in the wrong.
- yes: the comment says the poster did something wrong.
- no: the comment says the poster did nothing wrong, blames someone else, or takes no side.

Give a reason of one short sentence."""

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    response_mime_type="application/json",
    response_schema={
        "type": "object",
        "properties": {
            "llm_label": {"type": "string", "enum": ["yes", "no"]},
            "llm_reason": {"type": "string"},
        },
        "required": ["llm_label", "llm_reason"],
    },
)

In [ ]:
# Send one prompt, and try again after error 429 or 503.
def ask(prompt, config, max_tries=4):
    wait = 10
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config)
            time.sleep(6)  # stay under the free limit of calls per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in {429, 503} or attempt == max_tries:
                raise
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2

In [ ]:
# Label every comment in a table and return the labels next to the table.
# Each answer is saved to the file right away. If a cell that uses this stops, run it again: comments already in the file are skipped.
def label_all(table, path, config):
    path = Path(path)
    done = set(pd.read_csv(path)["id"]) if path.exists() else set()
    for row in table.itertuples():
        if row.id in done:
            continue
        answer = ask(f"Comment: {row.text}", config)
        line = pd.DataFrame([{"id": row.id, **answer, "model": GEMINI_MODEL}])
        line.to_csv(path, mode="a", header=not path.exists(), index=False)
        done.add(row.id)
        print(len(done), "of", len(table), "done")
    return pd.read_csv(path).merge(table, on="id")

In [ ]:
monday = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_reply_labels.csv")
known = with_verdict[with_verdict["id"].isin(monday["id"])]
gemini = label_all(known, "lab10_gemini_known.csv", CONFIG)  # takes about 3 minutes
len(gemini)  # Check: 20

## 4. Compare on the 20 comments with a verdict

In [ ]:
# Print the accuracy of always "no", the rule, and Gemini on these 20 comments. Check: 0.5 and 0.55. Gemini's number can change between runs

In [ ]:
# Make a confusion table for Gemini, like the one for the rule in part 2

In [ ]:
# Show the text, the answer, and Gemini's label and reason for the comments Gemini got wrong

Read the comments Gemini got wrong. Did Gemini misread them, or is the answer hard to see without the verdict word or the post?

Your answer:

## 5. Use Gemini on 20 comments with no verdict

Nothing gives the answer for these comments. The check in part 4 is your reason to trust the labels, or not.

In [ ]:
unknown = no_verdict.sample(n=20, random_state=211)
new = label_all(unknown, "lab10_gemini_unknown.csv", CONFIG)  # takes about 3 minutes
new[["text", "llm_label", "llm_reason"]]

In [ ]:
# How many of the 20 does Gemini label yes?

Read the 20 comments and Gemini's labels. How many labels do you disagree with?

Some of these comments take no side, such as a link to another subreddit. Every comment in the check took a side, so the check did not test these. How did Gemini label them?

Your answer:

## 6. Describe what you found

Write three or four sentences. Say how well Gemini did on the comments with a verdict, compared with always "no" and the rule. Then say whether that check is enough reason to trust its labels on comments with no verdict, and why.

Your finding:

Check that your key does not appear in any cell or output. Then save the notebook.

## If you have time: change the instruction and run it again

Change one thing in the definition. Label the 20 comments with a verdict again into a new file. Then compare the two runs.

In [ ]:
# A new SYSTEM2 and CONFIG2 with your change

In [ ]:
# Label known again with CONFIG2 into lab10_gemini_known_v2.csv

In [ ]:
# How many labels changed between the two runs, and Cohen's kappa between them
from sklearn.metrics import cohen_kappa_score